In [98]:
import pandas as pd
import random
import math
import numpy as np
from collections import Counter

In [2]:
df = pd.read_csv('./IMDB Dataset.csv')

In [5]:
tokens = list(map(lambda x: x.split(" "), df.review))
word_counter = Counter()
for review in tokens:
    for token in review:
        word_counter[token] -= 1

In [6]:
_ = word_counter.most_common()
vocab = list(set(map(lambda x: x[0], word_counter.most_common())))
len(vocab)

439838

In [7]:
word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

In [14]:
concatenated = list()
input_dataset = list()
for review in tokens:
    review_indices = list()
    for token in review:
        try:
            review_indices.append(word2index[token])
            concatenated.append(word2index[token])
        except:
            ""
    input_dataset.append(review_indices)
concatenated = np.array(concatenated)

random.shuffle(input_dataset)

In [24]:
def similar(target='beautiful', num=3):
    target_index = word2index[target]

    scores = Counter()
    for word, index in word2index.items():
        raw_difference = W0[index] - W0[target_index]
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))

    return scores.most_common(num)

In [17]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [44]:
lr, epochs = (.05, 2)
hidden_size, window, negative = 50, 2, 5

W0 = (np.random.rand(len(vocab), hidden_size) - 0.5) * 0.2
W1 = np.random.rand(len(vocab), hidden_size) * 0

layer_2_target = np.zeros(negative + 1)
layer_2_target[0] = 1

for review_i, review in enumerate(input_dataset * epochs):
    for target_i in range(len(review)):
        target_samples = [review[target_i]] + list(
            concatenated[(np.random.rand(negative) * len(concatenated)).astype('int').tolist()])

        left_context = review[max(0, target_i - window):target_i]
        right_context = review[target_i + 1: min(len(review), target_i + window)]

        layer_1 = np.mean(W0[left_context + right_context], axis=0)
        layer_2 = sigmoid(layer_1.dot(W1[target_samples].T))
        layer_2_delta = layer_2 - layer_2_target
        layer_1_delta = layer_2_delta.dot(W1[target_samples])

        W0[left_context + right_context] -= layer_1_delta * lr
        W1[target_samples] -= np.outer(layer_2_delta, layer_1) * lr

    if (review_i % 5000 == 0):
        print('\rProgress:' + str(review_i / float(len(input_dataset) * epochs)) + " " + str(similar('terrible')))

print()
print(similar('terrible', 10))

Progress:0.0 [('terrible', -0.0), ('psychotherapists,', -0.3738017053806138), ('"lair"', -0.377718743022532)]
Progress:0.05 [('terrible', -0.0), ('horrible', -1.7305494234273235), ('fantastic', -2.0546234780687875)]
Progress:0.1 [('terrible', -0.0), ('horrible', -2.435394986465902), ('fantastic', -2.722053473972546)]
Progress:0.15 [('terrible', -0.0), ('fantastic', -2.6547877497611747), ('horrible', -2.7425551823847067)]
Progress:0.2 [('terrible', -0.0), ('horrible', -2.8221620336544477), ('fantastic', -3.075296927418869)]
Progress:0.25 [('terrible', -0.0), ('horrible', -3.1819488112073397), ('dreadful', -3.29799849309155)]
Progress:0.3 [('terrible', -0.0), ('horrible', -2.578230805174594), ('lame', -3.4696563600757826)]
Progress:0.35 [('terrible', -0.0), ('horrible', -3.329129708134419), ('wonderful', -3.741547690245193)]
Progress:0.4 [('terrible', -0.0), ('horrible', -2.74019998467989), ('lousy', -3.5410089791305417)]
Progress:0.45 [('terrible', -0.0), ('horrible', -3.414681017357917

In [45]:
np.savez('./12_basic-nlp-3_weights.npz', W0=W0, W1=W1)
!ls ./12_basic-nlp-3_weights.npz

.rw-r--r--@ 352M  ./12_basic-nlp-3_weights.npz


In [51]:
data = np.load('./12_basic-nlp-3_weights.npz')
W0, W1 = data['W0'], data['W1']
del data

W0.shape, W1.shape

((439838, 50), (439838, 50))

## Код выше нужен чтобы у нас были веса. Взят из 11 главы.

-----------------------------------------

In [59]:
norms = np.sum(W0 * W0, axis=1)
norms.resize(norms.shape[0], 1)
normed_weights = W0 * norms

In [70]:
def make_sent_vect(words):
    indices = list(map(lambda x: word2index[x], filter(lambda x: x in word2index, words)))
    return np.mean(normed_weights[indices], axis=0)

In [71]:
review2vec = list()

for review in tokens:
    review2vec.append(make_sent_vect(review))

review2vec = np.array(review2vec)

In [73]:
def most_similar_reviews(review):
    v = make_sent_vect(review)

    scores = Counter()

    for i, val in enumerate(review2vec @ v):
        scores[i] = val

    most_similar = list()

    for idx, score in scores.most_common(3):
        most_similar.append(df.review[idx][0:40])

    return most_similar

In [83]:
most_similar_reviews(["boring", "awful"]), most_similar_reviews(["horrible", "bad"]), most_similar_reviews(
    ['beautiful', 'amazing'])

(['This was truly horrible. Bad acting, bad',
  "One of the worst movies I've ever seen!!",
  'If you thought "ROSEMARY\'S BABY" was bad'],
 ['This was truly horrible. Bad acting, bad',
  'Horrible waste of time - bad acting, plo',
  'I had high expectations for this movie a'],
 ['One of the funniest movies made in recen',
  'This was truly horrible. Bad acting, bad',
  "I'm surprised how many people give this "])

In [89]:
a = np.array([1, 2, 3])
b = np.array([0.1, 0.2, 0.3])
c = np.array([-1, -0.5, 0])
d = np.array([0, 0, 0])

identity = np.eye(3)
print(identity)

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [90]:
print(a @ identity)
print(b @ identity)
print(c @ identity)
print(d @ identity)

[1. 2. 3.]
[0.1 0.2 0.3]
[-1.  -0.5  0. ]
[0. 0. 0.]


In [92]:
this = np.array([2, 4, 6])
movie = np.array([10, 10, 10])
rocks = np.array([1, 1, 1])
print(this + movie + rocks)
print((this.dot(identity) + movie).dot(identity) + rocks)

[13 15 17]
[13. 15. 17.]


In [124]:
def softmax(x_):
    x = np.atleast_2d(x_)
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

In [112]:
np.random.seed(1)

word_vects = {}
word_vects['yankees'] = np.array([[0., 0., 0.]])
word_vects['bears'] = np.array([[0., 0., 0.]])
word_vects['braves'] = np.array([[0., 0., 0.]])
word_vects['red'] = np.array([[0., 0., 0.]])
word_vects['sox'] = np.array([[0., 0., 0.]])
word_vects['lose'] = np.array([[0., 0., 0.]])
word_vects['defeat'] = np.array([[0., 0., 0.]])
word_vects['beat'] = np.array([[0., 0., 0.]])
word_vects['tie'] = np.array([[0., 0.]])

sent2output = np.random.rand(3, len(word_vects))

identity = np.eye(3)

In [144]:
layer_0 = word_vects['red']
layer_l = layer_0.dot(identity) + word_vects['sox']
layer_2 = layer_l.dot(identity) + word_vects['defeat']

pred = softmax(layer_2.dot(sent2output))
pred

array([[0.11236815, 0.11242797, 0.11024658, 0.10988647, 0.11177167,
        0.11040224, 0.11019243, 0.11019328, 0.1125112 ]])

In [133]:
y = np.array([1, 0, 0, 0, 0, 0, 0, 0, 0])

pred_delta = pred - y
layer_2_delta = pred_delta.dot(sent2output.T)

defeat_delta = layer_2_delta * 1  # Пока что игнорим
layer_l_delta = layer_2_delta.dot(identity.T)

sox_delta = layer_l_delta * 1  # Пока что игнорим
layer_0_delta = layer_l_delta.dot(identity.T)

alpha = 0.01

word_vects['red'] -= layer_0_delta * alpha
word_vects['sox'] -= sox_delta * alpha
word_vects['defeat'] -= defeat_delta * alpha

identity -= np.outer(layer_0, layer_l_delta) * alpha
identity -= np.outer(layer_l, layer_2_delta) * alpha

sent2output -= np.outer(layer_2, pred_delta) * alpha